In [ ]:
# UNCOMMENT FOR INTERACTIVE PLOTTING
# %matplotlib notebook
%matplotlib widget

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation, rc

from snakelib import (
    FastSnake,
    RecurrentNeuralAgent,
    RecurrentPolicyAgent,
    run_episode,
    show_gui,
)
import utils

rc("animation", html="html5")

(ML:practical_work:genetic_snake)=
# Snake, sensors, and genetic learning

:::{admonition} Required files
:class: important
This notebook assumes that the following files are available in the same folder:

- {download}`snakelib.py <snakelib.py>`
- {download}`utils.py <utils.py>`
::: 

This practical session is organized in three parts:

1. **Discovering the game and its interface** (about 30 min)
2. **Designing a manual agent** from sensors, with optional memory (about 1 h 30)
3. **Training an automatic agent** with a recurrent neural network and a genetic algorithm

Throughout the practical, the observation space is intentionally restricted to three official sensors:

- `default`: tiny historical local sensor
- `compact`: compact topology-oriented sensor
- `local_lite`: small egocentric spatial view

The pedagogical goal is to compare three levels of design:

- a fully hand-written policy
- a manual policy enriched with explicit memory
- a parametric policy learned automatically

An important lesson of this tutorial is that **the most informative representation is not always the easiest one to optimize**.


## Part 1 - Discovering the game and its interface

![](https://www.pricepony.com.ph/blog/wp-content/uploads/2015/11/bestgameever-nokia-meme-628-300x300.jpg)

The goal of this first part is to understand:

- the game grid
- the available commands
- the relative direction convention
- the interactive display


In [ ]:
np.random.seed(0)
snake = FastSnake(
    Nrow=10, Ncol=10, max_snake_length=5
)  # TRY CHANGING MAX SNAKE LENGTH TO SEE HOW IT AFFECTS THE GAME

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
display(show_gui(snake, ax))

### 1. The grid

Each cell is indexed by a single integer. The border cells are walls, so the snake can only move inside the interior playable area.


In [ ]:
print(f"Grid dimensions: {snake.Nrow} rows, {snake.Ncol} columns, {snake.Ncell} cells")
utils.plot_grid(snake)

### 2. Commands

At each step, the snake does not choose an absolute direction. It chooses a **relative turn** with respect to its current heading:

- `-1`: turn right
- `0`: keep going forward
- `1`: turn left


In [ ]:
np.random.seed(0)
snake.reset()
snake.turn(-1)
snake.turn(0)
snake.turn(-1)
snake.turn(0)
snake.turn(1)

print("Current direction:", snake.get_current_direction())
print(
    "Reachable relative neighbor cells [right, front, left]:", snake.get_neighbors_pos()
)
utils.plot_snake(snake)

### Question 1

Starting from the initial state, which sequence of commands makes the snake eat the first fruit?

Write your answer in the next cell, then check it graphically.


In [ ]:
np.random.seed(0)
snake.reset()

# Write your command sequence here
# Example:
# snake.turn(-1)
# snake.turn(0)

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
display(show_gui(snake, ax))

### 3. Available sensors

![Chris Roberts (video game developer)](https://media.tenor.com/ziGwYdlteFoAAAAd/chris-roberts-video-game-designer.gif)

The module `snakelib.py` exposes three sensors for this practical. They all describe the same game state, but at different levels of abstraction.

#### `default` sensor

This sensor returns a vector of length `5`.

The first three values describe the three relative neighboring cells in the order `[right, front, left]`:

- `1.0`: the fruit is in that neighboring cell
- `0.0`: the neighboring cell is free
- `-1.0`: the neighboring cell is blocked by a wall or by the snake body

The last two values encode the fruit direction in the snake reference frame:

- component 4: sign of the fruit position along the **forward/backward** axis
- component 5: sign of the fruit position along the **right/left** axis

So `default` is extremely small and cheap, but it contains only immediate local information plus a coarse fruit direction.

#### `compact` sensor

This sensor returns a vector of length `21`. It analyzes the **three possible actions** `[right, front, left]`, not just the current neighboring cells.

For each action, it stores `6` values:

1. `safe`: `1` if the move does not die immediately, `-1` otherwise
2. `eats_fruit`: `1` if the move eats the fruit, `0` otherwise
3. `free_space_ratio`: fraction of the reachable free region after this move
4. `tail_reachable`: `1` if the head can still reach the tail after this move, `-1` otherwise
5. `fruit_path_score`: normalized score related to the path length from the next head position to the fruit
6. `tail_path_score`: normalized score related to the path length from the next head position to the tail

This gives `3 x 6 = 18` values. The last `3` values are global:

- normalized fruit direction along the forward/backward axis
- normalized fruit direction along the right/left axis
- normalized snake length

So `compact` is still small enough for genetic optimization, but much more informative than `default`.

#### `local_lite` sensor

This sensor returns a flattened vector of length `125`, which can be reshaped into `(5, 5, 5)`.

It is a **head-centered egocentric observation**:

- the snake head is always at the center of the `5 x 5` window
- the tensor is rotated so that the snake is always facing **upward** in the observation
- the first axis is the channel index

The `5` channels are:

1. walls
2. snake body, without the head
3. fruit
4. tail
5. region reachable from the head when the tail is considered free on the next step

This sensor preserves local geometry. It is richer than `compact`, but it also makes the learning problem larger.

In practice in this tutorial:


- `default` is the easiest to read and is also the easiest sensor to optimize automatically
- `compact` contains more useful structure for manual reasoning, but may be harder to optimize genetically
- `local_lite` preserves local geometry, but it is the most difficult of the three under a small CPU budget


In [ ]:
np.random.seed(0)
snake.reset()

print("default shape:", snake.sensors(method="default").shape)
print("compact shape:", snake.sensors(method="compact").shape)
print("local_lite shape:", snake.sensors(method="local_lite").shape)
print()
print("default:", snake.sensors(method="default"))
print()
print("compact:", snake.sensors(method="compact"))
print()
print("local_lite tensor shape:", snake.local_egocentric_channels_lite(radius=2).shape)

### Visualizing sensors

You can ask the graphical interface to display one sensor while the game is running.


In [ ]:
np.random.seed(0)
snake.reset()
snake.display_sensor_method = "compact"

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
display(show_gui(snake, ax))

## Part 2 - Designing a manual agent

In this part, you must build an agent that plays automatically **without learning**.

Your agent must be a Python function that receives:

- one observation vector
- an optional internal memory state

and returns:

- one move in `{-1, 0, 1}`
- an optional updated memory state

This is the core part of the practical. It forces you to understand what the sensors really contain and whether memory helps or not.


### Question 2

Write a function `manual_agent_step(observation, memory)` that returns a pair `(turn, new_memory)`.

Suggested workflow:

1. start with `sensor_method = "compact"` and `memory = None`
2. obtain a simple agent that survives and occasionally eats a fruit
3. then add explicit memory if you think it helps
4. finally compare with `local_lite`

Possible memory ideas:

- store the last few moves
- keep a temporary local mode or strategy
- count the number of turns since the last fruit


In [ ]:
SENSOR_METHOD = "compact"
TURN_IDS = np.array([-1, 0, 1])


def manual_agent_step(observation, memory):
    """
    Parameters
    ----------
    observation : ndarray
        Observation returned by snake.sensors(method=SENSOR_METHOD)
    memory : object
        Internal state chosen freely by the student

    Returns
    -------
    turn : int
        Must belong to {-1, 0, 1}
    new_memory : object
        Updated internal memory
    """

    # TODO: replace this baseline strategy
    turn = -1
    new_memory = memory
    return turn, new_memory


def play_manual_episode(
    sensor_method=SENSOR_METHOD, max_turns=200, seed=0, max_snake_length=None
):
    snake = FastSnake(Nrow=10, Ncol=10, max_snake_length=max_snake_length)
    snake.reset(fix_seed=seed)
    memory = None
    turns = 0

    while snake.status == 0 and turns < max_turns:
        observation = snake.sensors(method=sensor_method)
        turn, memory = manual_agent_step(observation, memory)
        snake.turn(int(turn))
        turns += 1

    return {
        "score": snake.score,
        "turns": turns,
        "status": snake.status,
        "memory": memory,
    }


play_manual_episode(max_snake_length=5)

### Animation of your manual agent

Use the next cell to observe the behavior of your strategy.


In [ ]:
snake_manual = FastSnake(Nrow=10, Ncol=10)
snake_manual.record_turns = True
snake_manual.recorded_sensors_method = SENSOR_METHOD
memory_manual = None


def update_manual(*args):
    global memory_manual
    if snake_manual.status == 0:
        observation = snake_manual.sensors(method=SENSOR_METHOD)
        turn, memory_manual = manual_agent_step(observation, memory_manual)
        snake_manual.turn(int(turn))
        im_manual.set_array(snake_manual.grid)
        title_manual.set_text(
            f"Score = {snake_manual.score} | Iteration = {snake_manual.iteration}"
        )
    else:
        title_manual.set_text(f"Game over | Score = {snake_manual.score}")
    return (im_manual,)


fig_manual, ax_manual = plt.subplots()
ax_manual.axis("off")
title_manual = ax_manual.set_title("Manual agent", animated=True)
im_manual = plt.imshow(snake_manual.grid, interpolation="nearest", animated=True)
anim_manual = animation.FuncAnimation(
    fig_manual, update_manual, frames=300, interval=50, blit=True
)
plt.close()
anim_manual

### Question 3

Build a small benchmark protocol to evaluate your manual strategy on several random seeds.

The minimum expected output is:

- a mean score
- a maximum score
- a mean number of turns

If you compare several sensors or memory designs, keep the results in a table.


In [ ]:
N_EPISODES = 20
manual_scores = []
manual_turns = []

for seed in range(N_EPISODES):
    stats = play_manual_episode(
        sensor_method=SENSOR_METHOD, max_turns=300, seed=seed, max_snake_length=None
    )
    manual_scores.append(stats["score"])
    manual_turns.append(stats["turns"])

manual_data = pd.DataFrame({"score": manual_scores, "turns": manual_turns})
manual_data.describe().loc[["mean", "std", "min", "max"]]

## Part 3 - Automatic learning with an RNN and a genetic algorithm

In this part, the hand-written decision function is replaced by a **recurrent neural network**.

At each step, the network receives:

- the current observation
- an internal memory vector `m_t`

and returns:

- three action scores
- a new memory vector `m_{t+1}`

Learning is performed **without gradients**, only by evolving a population of parameters.

### What is expected in this part

You are expected to do more than just run the code. Your work should include:

- a first successful training run with the **default** sensor
- at least one comparison involving memory size, for example `memory_size = 0` versus `memory_size = 4`
- at least one comparison involving the sensor, for example `default` versus `compact`
- plots of score and fitness across generations
- one visualization of the best learned agent
- a short discussion of what worked, what did not work, and why

### Important interpretation

In this tutorial, the main goal is **not** to prove that richer sensors or larger memory always improve performance. In fact, under a small CPU budget and with pure genetic optimization, the simplest setup is often the easiest one to train.

So Part 3 should be read as an experiment on the trade-off between:

- information quality
- model size
- optimization difficulty
- computational budget


### Useful helper function

We keep the model intentionally small:

- one small hidden layer
- an explicit low-dimensional memory
- selection + crossover + Gaussian mutation

The next function evaluates one individual of the population on several episodes and returns its mean fitness.

For this part, the recommended order is:

1. start with the easiest automatic setup: `sensor_method = "default"`, `memory_size = 0`
2. test whether memory helps on the same sensor
3. only then compare with a richer but harder sensor such as `compact`
4. treat `local_lite` as an optional extension

This way, students first obtain a working baseline, then analyze why richer observations do not automatically lead to better learning.


In [ ]:
def evaluate_recurrent_weights(
    weights,
    sensor_method="default",
    memory_size=0,
    hidden_sizes=(8,),
    episodes=3,
    max_turns=300,
    seed=0,
    max_snake_length=None,
):
    snake = FastSnake(Nrow=10, Ncol=10, max_snake_length=max_snake_length)
    observation_size = snake.sensors(method=sensor_method).size

    network = RecurrentNeuralAgent(
        weights=weights,
        observation_size=observation_size,
        memory_size=memory_size,
        hidden_sizes=hidden_sizes,
        action_size=3,
    )
    agent = RecurrentPolicyAgent(network, sensor_method=sensor_method)

    scores = []
    turns = []
    for episode_id in range(episodes):
        stats = run_episode(
            snake,
            agent,
            max_turns=max_turns,
            fix_seed=seed + episode_id,
        )
        scores.append(stats["score"])
        turns.append(stats["turns"])

    score = np.mean(scores)
    turn = np.mean(turns)
    fitness = score * 1000.0 - 0.1 * turn
    return {
        "score": score,
        "turns": turn,
        "fitness": fitness,
    }

### Question 4

Complete and run the following genetic workflow.

The workflow is intentionally split into two cells:

1. an **initialization/reset** cell
2. a **training** cell that can be re-run to continue evolution

### Required progression

1. start with `sensor_method = "default"` and `memory_size = 0`
2. start with `max_snake_length = 2`.
2. obtain a first baseline learning curve
3. test the effect of memory on the same sensor, for example by switching to `memory_size = 4`
4. only then move to a richer sensor such as `compact`
5. treat `local_lite` as optional

### Minimum expected output

- one baseline run with the default sensor
- one comparison between two memory sizes on the default sensor
- one comparison between `default` and `compact`
- the score plot
- the fitness plot
- one animation of the best learned agent
- a short written conclusion

### Important note

In this tutorial, it is normal if the richer sensors do **not** outperform the default sensor. This does not mean that they are bad sensors. It means that a richer representation can also create a harder optimization problem for the genetic algorithm.

### Questions to discuss

- Does a richer sensor always help?
- Does a larger memory always help?
- What limits the maximum score ?
- What is the effect of the `max_snake_length` variable ?
- With a fixed CPU budget, is it better to use a larger population or more generations?
- Is it easier to improve the model by changing the sensor, the memory, or the genetic hyperparameters?
- Why can the simplest sensor be the most effective one in this setup?


In [ ]:
sensor_method = "default"
memory_size = 0
hidden_sizes = (16,)

Npop = 100
Ngen = 12
Nepisodes = 3
max_turns = 500
keep_ratio = 0.2
mutation_ratio = 0.3
mutation_sigma = 0.15
force_reset = True
max_snake_length = 5

probe_snake = FastSnake(Nrow=10, Ncol=10, max_snake_length=max_snake_length)
observation_size = probe_snake.sensors(method=sensor_method).size
nweights = RecurrentNeuralAgent.count_weights(
    observation_size=observation_size,
    memory_size=memory_size,
    hidden_sizes=hidden_sizes,
    action_size=3,
)

keep_count = max(2, int(keep_ratio * Npop))

needs_reset = (
    force_reset
    or "all_weights" not in globals()
    or all_weights.shape != (Npop, nweights)
    or "fitness" not in globals()
    or fitness.shape != (Npop,)
    or "scores" not in globals()
    or scores.shape != (Npop,)
    or "turns" not in globals()
    or turns.shape != (Npop,)
    or "best_score_history" not in globals()
    or "best_fitness_history" not in globals()
)

if needs_reset:
    all_weights = np.random.normal(loc=0.0, scale=0.5, size=(Npop, nweights))
    fitness = np.zeros(Npop)
    scores = np.zeros(Npop)
    turns = np.zeros(Npop)
    best_score_history = []
    best_fitness_history = []
    print("Initialized a new population.")
else:
    print("Population kept unchanged. You can continue training in the next cell.")

In [ ]:
start_generation = len(best_score_history)

for generation in range(Ngen):
    current_generation = start_generation + generation

    for i in range(Npop):
        metrics = evaluate_recurrent_weights(
            all_weights[i],
            sensor_method=sensor_method,
            memory_size=memory_size,
            hidden_sizes=hidden_sizes,
            episodes=Nepisodes,
            max_turns=max_turns,
            max_snake_length=max_snake_length,
            seed=1000 * current_generation + 10 * i,
        )
        fitness[i] = metrics["fitness"]
        scores[i] = metrics["score"]
        turns[i] = metrics["turns"]

    order = np.argsort(fitness)[::-1]
    best_score_history.append(scores[order[0]])
    best_fitness_history.append(fitness[order[0]])

    print(
        f"Generation {current_generation + 1:02d} | best score = {scores[order[0]]:.2f} | best fitness = {fitness[order[0]]:.2f}"
    )

    new_weights = np.zeros_like(all_weights)
    new_weights[:keep_count] = all_weights[order[:keep_count]]

    for i in range(keep_count, Npop):
        parents = np.random.choice(keep_count, size=2, replace=False)
        alpha = np.random.rand(nweights)
        child = new_weights[parents[0]] * alpha + new_weights[parents[1]] * (
            1.0 - alpha
        )
        if np.random.rand() < mutation_ratio:
            child += np.random.normal(loc=0.0, scale=mutation_sigma, size=nweights)
        new_weights[i] = child

    all_weights[:] = new_weights

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(best_score_history, "o-", color="tab:blue")
axes[0].set_title("Best score")
axes[0].set_xlabel("Generation")
axes[0].set_ylabel("Score")
axes[0].grid()

axes[1].plot(best_fitness_history, "s-", color="tab:orange")
axes[1].set_title("Best fitness")
axes[1].set_xlabel("Generation")
axes[1].set_ylabel("Fitness")
axes[1].grid()

plt.tight_layout()
plt.show()

### Visualizing the best learned agent


In [ ]:
best_weights = all_weights[np.argmax(fitness)]

best_network = RecurrentNeuralAgent(
    weights=best_weights,
    observation_size=observation_size,
    memory_size=memory_size,
    hidden_sizes=hidden_sizes,
    action_size=3,
)
best_agent = RecurrentPolicyAgent(best_network, sensor_method=sensor_method)

snake_learned = FastSnake(Nrow=10, Ncol=10, max_snake_length=max_snake_length)


def update_learned(*args):
    if snake_learned.status == 0:
        snake_learned.turn(best_agent(snake_learned))
        im_learned.set_array(snake_learned.grid)
        title_learned.set_text(
            f"Score = {snake_learned.score} | Iteration = {snake_learned.iteration}"
        )
    else:
        title_learned.set_text(f"Game over | Score = {snake_learned.score}")
    return (im_learned,)


best_agent.reset()
fig_learned, ax_learned = plt.subplots()
ax_learned.axis("off")
title_learned = ax_learned.set_title("Learned recurrent agent", animated=True)
im_learned = plt.imshow(snake_learned.grid, interpolation="nearest", animated=True)
anim_learned = animation.FuncAnimation(
    fig_learned, update_learned, frames=400, interval=40, blit=True
)
plt.close()
anim_learned

## Summary

At the end of the practical, you should be able to discuss:

- the choice of sensor (`default`, `compact`, `local_lite`)
- whether explicit memory is useful or not
- the difference between a hand-written strategy and a learned one
- why a smaller representation can be easier to optimize than a richer one
- the effect of population size and mutation
- the limits of pure genetic learning on Snake
